In [107]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [108]:
train = pd.read_csv('train.csv').drop('id', axis=1)
test = pd.read_csv('test.csv').drop('id', axis=1)

In [109]:
train.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.20,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35,No


In [110]:
train.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.000000,594194.000000
mean,0.114102,36.577258,65.866223,2494.377057
std,0.317936,25.061922,31.067444,2353.916710
min,0.000000,1.000000,18.250000,18.800000
25%,0.000000,12.000000,29.900000,639.650000
50%,0.000000,35.000000,74.100000,1433.650000
75%,0.000000,62.000000,90.800000,4263.800000
max,1.000000,72.000000,118.750000,8684.800000


In [111]:
train.select_dtypes(include='number').columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

In [112]:
for df in [train, test]:

    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

    df['MCSquared'] = df['MonthlyCharges'] ** 2
    df['TCSquared'] = df['TotalCharges'] ** 2

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    df['MCSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['TCSqrt'] = np.sqrt(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    # Loyalty
    df["TenureLog"] = np.log1p(df["tenure"])
    df["TenureSquared"] = df["tenure"]**2
    df['TenureSqrt'] = np.sqrt(df['tenure'])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    # Spending
    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]

    # Contract
    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    # Payment
    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    # Service Engagement
    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    # Internet
    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # Early Risk
    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    # Value
    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

    # df['TenureGT3'] = df['tenure'].gt(3.0).astype(int)
    # df['TenureGT6'] = df['tenure'].gt(6.0).astype(int)
    # df['TenureGT12'] = df['tenure'].gt(12.0).astype(int)
    # df['TenureGT24'] = df['tenure'].gt(24.0).astype(int)
    # df['TenureGT36'] = df['tenure'].gt(36.0).astype(int)

    # df['MCGT20'] = df['MonthlyCharges'].gt(20.0).astype(int)
    # df['MCGT40'] = df['MonthlyCharges'].gt(40.0).astype(int)
    # df['MCGT60'] = df['MonthlyCharges'].gt(60.0).astype(int)

    # df['TCGT500'] = df['TotalCharges'].gt(500.0).astype(int)
    # df['TCGT1000'] = df['TotalCharges'].gt(1000.0).astype(int)
    # df['TCGT4000'] = df['TotalCharges'].gt(4000.0).astype(int)
    # df['TCGT8000'] = df['TotalCharges'].gt(8000.0).astype(int)

/var/folders/sk/bf5_j5395pn851dn2wx13dy80000gn/T/ipykernel_6952/2404071475.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)
/var/folders/sk/bf5_j5395pn851dn2wx13dy80000gn/T/ipykernel_6952/2404071475.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which 

In [113]:
train['MonthlyMinusMedian'] = train['MonthlyCharges'] - train['MonthlyCharges'].median()
test['MonthlyMinusMedian'] = test['MonthlyCharges'] - train['MonthlyCharges'].median()

train['TenureGTMedian'] = train['tenure'].gt(train['tenure'].median()).astype(int)
test['TenureGTMedian'] = test['tenure'].gt(train['tenure'].median()).astype(int)

train['MCGTMedian'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].median()).astype(int)
test['MCGTMedian'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].median()).astype(int)

train['MCGTMean'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].mean()).astype(int)
test['MCGTMean'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].mean()).astype(int)

train['TCGTMedian'] = train['TotalCharges'].gt(train['TotalCharges'].median()).astype(int)
test['TCGTMedian'] = test['TotalCharges'].gt(train['TotalCharges'].median()).astype(int)

train['TCGTMean'] = train['TotalCharges'].gt(train['TotalCharges'].mean()).astype(int)
test['TCGTMean'] = test['TotalCharges'].gt(train['TotalCharges'].mean()).astype(int)

train['MCGT25%'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.25)).astype(int)
test['MCGT25%'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.25)).astype(int)

train['MCGT75%'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)
test['MCGT75%'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)

train['TCGT25%'] = train['TotalCharges'].gt(train['TotalCharges'].quantile(0.25)).astype(int)
test['TCGT25%'] = test['TotalCharges'].gt(train['TotalCharges'].quantile(0.25)).astype(int)

train['TCGT75%'] = train['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)
test['TCGT75%'] = test['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)

In [114]:
X, y = train.drop('Churn', axis=1), train['Churn']
y = y.map({'Yes': 1, 'No': 0})

In [115]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

categorical_features = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
     transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

preprocessor.set_output(transform='pandas')

X_encoded = preprocessor.fit_transform(X)
test_encoded = preprocessor.transform(test)

In [116]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=43
)

oof_preds = np.zeros(len(X_encoded))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_encoded.iloc[train_idx], X_encoded.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        random_state=43,
        eval_metric="auc"
    )

    model.fit(X_train, y_train)
    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(test_encoded)[:, 1] / skf.n_splits

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [117]:
from sklearn.metrics import roc_auc_score

cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9162195329951571


In [119]:
sample_submission = pd.read_csv('sample_submission.csv')
submission = sample_submission.copy()
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)